This notebook implements GRPO from scratch in `PyTorch`, without relying on `trl` or `verl`.
The goal was to train `Qwen2.5-1.5B` on GSM8K and evaluate generalization on GSM-Hard (OOD).

Along the way: 
- entropy collapse with beta=0,
- OOM from materializing [B,L,V] logits,
- conflicts between LoRA and gradient checkpointing,
- and failed attempts to speed up generation with flash-attention and unsloth.

**Results:** 30/56% (soft/strict matching) to 70% on GSM8K, 13/27% to 37% on GSM-Hard. All within ~1-2hrs on RTX 5090.

In [ ]:
# Here are all needed libs, I run it on vast.ai RTX 3090/5090 pods
# import sys
# !{sys.executable} -m pip install --upgrade transformers peft bitsandbytes trl datasets accelerate huggingface_hub python-dotenv tensorboard wandb unsloth matplotlib

In [ ]:
# to speed-up LLM - I couldn't build it in 1hr :(
# import sys
# !{sys.executable} -m pip install flash-attn --no-build-isolation

# Setup

## Basic setup (config, hf_login, logger, etc)

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" # to see backtrace of random cuda errors

In [ ]:
# from huggingface_hub import login
# hf_token = "your_hf_token"
# login(token=hf_token)

In [ ]:
import logging

logger = logging.getLogger("grpo")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(funcName)s | %(message)s")

console = logging.StreamHandler()
console.setFormatter(fmt)

file = logging.FileHandler("train.log")
file.setFormatter(fmt)

logger.addHandler(console)
logger.addHandler(file)

In [ ]:
# unsloth should be imported fisrt, but it didn't speed up anything yet imposed some restrictions (e.g. lora_dropout)
# import unsloth 
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from tqdm import tqdm
import time
import math
from peft import get_peft_model, LoraConfig

In [ ]:
class Config:
    # model
    model_name = "Qwen/Qwen2.5-1.5B"
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    lora_rank = 128
    lora_alpha = 128 * 2
    dtype = torch.bfloat16
    lora_dropout = 0.05 # should be .0 if unsloth
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    max_prompt_length = 256
    max_new_tokens = 512

    # grpo
    group_size = 16
    accumulation_steps = 4
    eps = 10.0
    beta = 5e-4 # model collapses without it (and with no scheduler)
    lr = 3e-5
    weight_decay = 5e-3
    num_updates = 300 # will be multiplied on accumulation_steps 
    clip_grad_norm = 1.0

    # eval & logging, numbers are in # updates
    eval_metrics_steps = 15
    eval_generation_steps = 15
    checkpoint_steps = 50
    inference_batch_size = 128 # tune it for your GPU - the more the faster
    log_dir = "/workspace/logs"

In [ ]:
config = Config()

## Some stuff for memory management (hi, OOM)

In [ ]:
import gc
import ctypes

def cleanup_cuda(globs=None, names=None):
    if globs is None:
        globs = globals()

    if names is None:
        names = [
            "lp", "old_lp", "ref_lp", "loss", "policy_loss", "kl_loss",
            "old_ratio", "ref_ratio", "A", "rewards", "trajectories",
            "attention_mask", "input_ids", "completions"
        ]

    # detach all visible CUDA tensors
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                try:
                    obj.detach_()
                except:
                    pass
        except:
            pass

    # delete selected globals
    for name in names:
        if name in globs:
            del globs[name]

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

    logger.info(f"allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    logger.info(f"reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    logger.info("Next backward may break since we cleaned some tensors")

In [ ]:
def mem(prefix="Allocated memory"):
    msg = f"{prefix}: {round(torch.cuda.memory_allocated() / 1024**3, 3)} GB"
    logger.info(msg)

In [ ]:
logger.setLevel(logging.INFO)
mem("You won't see me until you set logger.setLevel(logging.INFO)")

## `transformers` model & tokenizer

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(config.model_name, pad_token="<|extra_0|>")

# model = AutoModelForCausalLM.from_pretrained(
#     config.model_name,
#     device_map=str(config.device),
#     dtype=config.dtype,
#     low_cpu_mem_usage=True,
#     offload_state_dict=True,
#     attn_implementation="sdpa", # I couldn't set up "flash_attention_2"
# )

# lora_config = LoraConfig(
#     r=config.lora_rank,
#     lora_alpha=config.lora_alpha,
#     target_modules=config.target_modules,
#     lora_dropout=config.lora_dropout,
#     task_type="CAUSAL_LM",
# )

# model = get_peft_model(model, lora_config)

# # Don't use it with unsloth
# model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
# model.generation_config.cache_implementation = "static" # it is important

# # additional stuff for padding
# tokenizer.add_special_tokens({'pad_token': '<|extra_0|>'}) # tokenizer.pad_token = "<|extra_0|>"
# model.config.pad_token_id = tokenizer.pad_token_id
# model.generation_config.pad_token_id = tokenizer.pad_token_id
# model.config.use_cache = False 
# tokenizer.padding_side, tokenizer.pad_token_id

## Load checkpoint

In [ ]:
from peft import PeftModel
model_name = config.model_name.split("/")[1]
model_name_hf = model_name + "grpo_gsm8k"

tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True, trust_remote_code=False,)
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    device_map=str(config.device),
    dtype=config.dtype,
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(
    model, 
    f"zinchse/{model_name_hf}", 
    # revision="8ac2281c4bd37ea1dd4b991f4f9574214dde43c0", # you may remove it, it is ~1000/1200 steps, 250/300 updates.
)

# Don't use it with unsloth
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.generation_config.cache_implementation = "static" # it is important

tokenizer.add_special_tokens({'pad_token': '<|extra_0|>'}) # tokenizer.pad_token = "<|extra_0|>"
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False 
tokenizer.padding_side, tokenizer.pad_token_id

## Helper to close `vastai` pod

In [ ]:
import requests

def finish_vastai_pod(vastai_api_key):
    resp = requests.get(
        "https://console.vast.ai/api/v0/instances/",
        headers={"Authorization": f"Bearer {vastai_api_key}"}
    )
    instances = resp.json()["instances"]
    
    for inst in instances:
        requests.put(
            f"https://console.vast.ai/api/v0/instances/{inst['id']}/",
            headers={"Authorization": f"Bearer {vastai_api_key}"},
            json={"state": "stopped"}
        )

# Data
I use `gsm8k` for train and `gsm-hard` for evals. Note, I used `<MYTHINK>` token to make the effect clear

In [ ]:
def process_gsm8k(x):
    x["prompt"] = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves "
        "it. The assistant first thinks about the reasoning process in the mind and then provides the "
        "user with the answer. The reasoning process and answer are enclosed within <MYTHINK>...</MYTHINK> "
        "and <answer>...</answer> tags, respectively, i.e.,\n"
        "<MYTHINK>\nreasoning process here\n</MYTHINK>\n"
        "<answer>\nanswer here\n</answer>\n"
        "User: " + x["question"] + "\nAssistant:"
    )
    x["answer"] = x["answer"].split("\n#### ")[1].replace(",", "")
    return x

def process_gsm_hard(x):
    x["prompt"] = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves "
        "it. The assistant first thinks about the reasoning process in the mind and then provides the "
        "user with the answer. The reasoning process and answer are enclosed within <MYTHINK>...</MYTHINK> "
        "and <answer>...</answer> tags, respectively, i.e.,\n"
        "<MYTHINK>\nreasoning process here\n</MYTHINK>\n"
        "<answer>\nanswer here\n</answer>\n"
        "User: " + x["question"] + "\nAssistant:"
    )
    x["answer"] = x["target"]
    return x
    

In [ ]:
from datasets import load_dataset

# NB: we dont pad data here

gsm8k = load_dataset("openai/gsm8k", "main")
gsm8k = gsm8k.map(process_gsm8k)
gsm8k = gsm8k.map(lambda x: tokenizer(x["prompt"], truncation=True, max_length=config.max_prompt_length, padding=False), batched=True)

gsm_hard = load_dataset("reasoning-machines/gsm-hard")["train"]
gsm_hard = gsm_hard.map(lambda x: {"question": x["input"]})
gsm_hard = gsm_hard.map(process_gsm_hard)
gsm_hard = gsm_hard.remove_columns(["input", "code", "target"])
gsm_hard = gsm_hard.map(lambda x: tokenizer(x["prompt"], truncation=True, max_length=config.max_prompt_length, padding=False), batched=True)

In [ ]:
# small subsets for evaluation
gsm8k_small = gsm8k['test'].select(range(config.inference_batch_size))
gsm_hard_small = gsm_hard.select(range(config.inference_batch_size))

# Rewards
Note, in `gsmk` we abuse the fact that all answer are integers. In `gsmk_hard` it is not the case anymore so we process it in a little bit different way.

In [ ]:
import re

def extract_llm_answer(completion) -> str:
    """extracts an integer inside the <answer> block assuming the right format"""
    try:
      return str(int(float(completion.split("<answer>")[1].split("</answer>")[0].strip())))
    except Exception:
      return ""

def extract_llm_answer_soft(completion):
    """extracts last integer from completion"""
    numbers = re.findall(r'-?\d+', completion)
    try:
        return str(int(float(numbers[-1]))) if numbers else None
    except Exception: 
        logger.info(numbers)
        return None

def is_correct(completions, answer, soft_answer=False, **kwargs):
    if soft_answer:
        return torch.tensor([2.0 * (extract_llm_answer_soft(c) == a) for c, a in zip(completions, answer)])
    else:
        return torch.tensor([2.0 * (extract_llm_answer(c) == a) for c, a in zip(completions, answer)])

def is_number(completions, **kwargs):
    return torch.tensor([0.5 * (len(extract_llm_answer(c)) > 0) for c in completions])

def has_right_format(completions, **kwargs):
    pattern = r"<MYTHINK>.*?</MYTHINK>\s*<answer>.*?</answer>" # allows multi-line reasoning
    return torch.tensor([0.5 if re.search(pattern, c, re.DOTALL) else 0.0 for c in completions])

def has_tags(completions, **kwargs):
    res = []
    for c in completions:
        score = 0.0
        if c.count("<MYTHINK>\n") == 1:
            score += 0.125
        if c.count("\n</MYTHINK>\n") == 1:
            score += 0.125
        if c.count("\n<answer>\n") == 1:
            score += 0.125
        if c.count("\n</answer>") == 1:
            score += 0.125
            trailing = c.split("</answer>")[-1].strip()
            score -= len(trailing) * 0.01 # some penalty
        res.append(score)
    return torch.tensor(res)

REWARD_FUNCS = [is_correct, is_number, has_right_format, has_tags]

For evals on `gsm_hard` we need separated processing of float answers:

In [ ]:
def extract_last_float(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    if not nums:
        return None
    return float(nums[-1])

def extract_gsm_hard_answer(completion):
    """strict: extracts float from <answer> block"""
    try:
        return float(completion.split("<answer>")[1].split("</answer>")[0].strip().replace(",", ""))
    except Exception:
        return None

def is_correct_gsm_hard(completions, answer, soft_answer=True, abs_tol=1e-6):
    out = []
    for pred_text, a in zip(completions, answer):
        pred = extract_last_float(pred_text) if soft_answer else extract_gsm_hard_answer(pred_text)
        ok = pred is not None and math.isclose(pred, float(a), rel_tol=0.0, abs_tol=abs_tol)
        out.append(float(ok))
    return torch.tensor(out)

# Utils

## Evaluation / Inference

In [ ]:
@torch.no_grad()
def infer_batch(model, batch, ref=False):
    """
    - input is a list of examples
    - ref=True if we inference reference (disabled lora)
    """
    was_training = model.training
    model.eval()
    
    lora_was_enabled = model.get_model_status().enabled
    if ref:
        model.disable_adapter_layers()

    input_ids_list = [ex["input_ids"] for ex in batch]
    max_len = max(len(ids) for ids in input_ids_list)

    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long, device=model.device)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long, device=model.device)

    for i, ids in enumerate(input_ids_list):
        ids = torch.tensor(ids, device=model.device)
        input_ids[i, max_len - len(ids):] = ids
        attention_mask[i, max_len - len(ids):] = 1

    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=config.max_new_tokens,
        use_cache=True,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    completions = tokenizer.batch_decode(outputs[:, max_len:], skip_special_tokens=True)

    if ref and lora_was_enabled:
        model.enable_adapter_layers()

    if was_training:
        model.train()

    return completions

Later we will use the next illustrative questions from the `gsm_hard`) for tracking the progress

In [ ]:
gsm_hard_samples_indices = [43, 44, 545]  
gsm_hard_samples = gsm_hard.select(gsm_hard_samples_indices)

# completions = infer_batch(model, gsm_hard_samples)
# for i, (idx, completion) in enumerate(zip(gsm_hard_samples_indices, completions)):
#     q, a = gsm_hard[idx]['question'], gsm_hard[idx]['answer']
#     print(f"**Question:**\n{q}\n\n**Answer:**\n{a}\n\n**Completion (untrained model):**\n{completion}")

I use only greedy evaluation because group-generation is the bottleneck. 

In [ ]:
@torch.no_grad()
def evaluate_gsm8k(model, eval_ds, tokenizer, reward_funcs, soft_answer=False, batch_size=config.inference_batch_size):
    """
    batched version of greedy evaluation on exact format match (soft_answer=False)
    or on match of the last written digit with the expected answer (soft_answer=True)
    """
    was_training = model.training
    model.eval()
    
    all_rewards = []
    all_accuracies = []
    for start in tqdm(range(0, len(eval_ds), batch_size), desc="Eval GSM-8k"):
        batch = eval_ds.select(range(start, min(start + batch_size, len(eval_ds))))
        completions = infer_batch(model, batch)
        rewards = torch.stack([f(completions, answer=batch["answer"]) for f in reward_funcs]).sum(dim=0)
        acc = is_correct(completions, answer=batch["answer"], soft_answer=soft_answer) / 2.0
        all_rewards.append(rewards)
        all_accuracies.append(acc)
    
    if was_training:
        model.train()
    
    return torch.cat(all_rewards).mean().item(), torch.cat(all_accuracies).mean().item()

In `gsm_hard` answer might be float, we process it by `is_correct_gsm_hard`. Note, here by default `soft_answer=True`

In [ ]:
@torch.no_grad()
def evaluate_gsm_hard(model, eval_ds, tokenizer, batch_size=config.inference_batch_size, soft_answer=True):
    model.eval()
    all_accuracies = []
    for start in tqdm(range(0, len(eval_ds), batch_size), desc="Eval GSM-Hard"):
        batch = eval_ds.select(range(start, min(start + batch_size, len(eval_ds))))
        completions = infer_batch(model, batch)
        acc = is_correct_gsm_hard(completions, answer=batch["answer"], soft_answer=soft_answer)
        all_accuracies.append(acc)
    model.train()
    return torch.cat(all_accuracies).mean().item()

## Main steps of the training loop

In [ ]:
def generate_group(pi, ex, max_new_tokens, group_size):
    was_training = pi.training
    pi.eval()

    # use repeat instead of expand for backprop & reuse mem by as_tensor
    input_ids = torch.as_tensor(ex["input_ids"], device=pi.device).unsqueeze(0).repeat(group_size, 1)
    attention_mask = torch.tensor(ex["attention_mask"]).unsqueeze(0).repeat(group_size, 1).to(pi.device)
    
    with torch.no_grad():
        mem("generate_group: Before generation")
        t0 = time.time()
        trajectories = pi.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, use_cache=True, do_sample=True)
        logger.info(f"generation time: {time.time()-t0:.1f}s")
    
    if was_training:
        pi.train()
    
    return trajectories

In [ ]:
import torch.nn.functional as F
def get_logprobs_and_entropy(model, trajectories, attention_mask, prompt_length, chunk_size=128):
    """
    - all padded tokens will have zero
    - uses fused log_softmax with gather to not materialize [B,T,V] during the backward (at some point it fixed OOM)
    """    
    mem("before prefill")
    out = model(trajectories, attention_mask=attention_mask)

    # artifact from a model convergence debug
    # for i in [0, 3, 5, 7]:
    #     sample_logits = out.logits[0, prompt_length + 1 + i, :]
    #     probs = torch.softmax(sample_logits, dim=-1)
    #     top5 = torch.topk(probs, 5)
    #     logger.info(f"top5 probs: {top5.values}")
    #     logger.info(f"max entropy: {torch.log(torch.tensor(float(probs.shape[0])))}")

    mem("before chunked log_softmax")
    targets = trajectories[:, prompt_length:]
    mask = attention_mask[:, prompt_length:]
    log_probs = torch.zeros(targets.shape, device=model.device)
    entropy_acc = torch.zeros(targets.shape, device=model.device)
    for i in range(0, targets.shape[1], chunk_size):
        j = min(i + chunk_size, targets.shape[1])
        chunk_logits = out.logits[:, prompt_length-1+i:prompt_length-1+j, :]
        lsm = F.log_softmax(chunk_logits, dim=-1)
        log_probs[:, i:j] = lsm.gather(-1, targets[:, i:j].unsqueeze(-1)).squeeze(-1)
        entropy_acc[:, i:j] = -(lsm.exp() * lsm).sum(dim=-1)
        del chunk_logits, lsm
        
    mem("after chunked log_softmax")
    entropy = ((entropy_acc * mask).sum(dim=-1) / mask.sum(dim=-1)).mean()
    del entropy_acc, out
    
    return log_probs * mask, entropy

**Attention.** I think, theory-wise it is the core cell of the GRPO. Note, here I use rollout = 1 (i.e. each sampled trajectory is used only once to update the model). So we don't actually need the clipping because $\pi_{old}$ is always the same with $\pi$ and old_ratio is equal to one. This observations allows us also to **reduce one generation loop** - we can simply `detach` the logprobs from the $\pi$ and use them as logprobs of $\pi_{old}$.

PS I also tried to set $\beta = 0$, remove KL-div term and avoid one more generation, but it has led to the model divergence. So I decided to keep it here.

In [ ]:
def calculate_metrics(pi, trajectories, ex, eps, beta, accumulation_steps):
    """
    - all things are calculated per-trajectory
    - paddings are zero, so they don't affect ratios
    - rollout == 1 => old_lp = lp.detach() => old_ratio == 1
    - I use per-token KL clipping to give model a chance to learn 
      from other tokens, otherwise <MYTHINK> might broke everything.
    - If beta is zero (default value), I don't calculate KL at all
    """
    prompt_length, group_size = len(ex["input_ids"]), trajectories.shape[0]

    attention_mask = torch.as_tensor(trajectories != tokenizer.pad_token_id, dtype=torch.long, device=pi.device)
    lp, entropy = get_logprobs_and_entropy(pi, trajectories, attention_mask, prompt_length)
    old_lp = lp.detach()  # single update per rollout
    with torch.no_grad():
        with pi.disable_adapter():
            ref_lp, _ = get_logprobs_and_entropy(pi, trajectories, attention_mask, prompt_length)

    completions = tokenizer.batch_decode(trajectories[:, prompt_length:], skip_special_tokens=True)
    rewards = torch.stack([f(completions, answer=[ex["answer"]] * group_size) for f in REWARD_FUNCS]).to(pi.device).sum(dim=0)
    A = rewards - rewards.mean(dim=0)
    A = (A / (A.std(dim=0) + 1e-3)).clamp(-5, 5) # let's check if it helps
    
    old_ratio = torch.exp((lp - old_lp).sum(dim=-1))
    policy_loss = -torch.min(old_ratio * A, torch.clamp(old_ratio, 1 - eps, 1 + eps) * A).sum()

    n_real_tokens = attention_mask[:,prompt_length:].sum(dim=-1, keepdim=True)
    
    mem("before ref_ratio")
    if beta:
        ref_ratio = torch.exp(ref_lp - lp).clamp(1e-4, 1e4)
        per_token_kl_loss = ((ref_ratio - torch.log(ref_ratio) - 1) / n_real_tokens).sum()
        assert len(ref_ratio.shape) == 2 and ref_ratio.shape[0] == group_size, ("Something went wrong:", ref_ratio.shape)
        assert len(n_real_tokens.shape) == 2 and n_real_tokens.shape == (group_size, 1), ("Something went wrong:", n_real_tokens.shape)
    else:
        per_token_kl_loss = torch.tensor(.0, device=pi.device)
        
    loss = (policy_loss + beta * per_token_kl_loss) / (accumulation_steps * group_size)

    return {
        "loss": loss,
        "policy_loss": policy_loss,
        "kl_loss": per_token_kl_loss,
        "entropy": entropy,
        "rewards": rewards,
        "advantage": A,
        "old_ratio": old_ratio,
        "completions": completions,
        "lp": lp,
        "old_lp": old_lp,
        "ref_lp": ref_lp,
    }

## Routines for logging

In [ ]:
def maybe_do_opt_step(pi, opt, scheduler, loss, step, accumulation_steps, clip_grad_norm):
    loss.backward()
    if not step % accumulation_steps:
        torch.nn.utils.clip_grad_norm_(pi.parameters(), max_norm=clip_grad_norm)
        opt.step()
        opt.zero_grad()    
        scheduler.step()

In [ ]:
def log_metrics(metrics, step, writer, accumulation_steps):
    logger.warning(f"REWARDS {metrics["rewards"].tolist()} | {metrics["rewards"].mean().item()}")
    writer.add_scalar("train/loss", metrics["loss"].item() * accumulation_steps, step)
    writer.add_scalar("train/policy_loss", metrics["policy_loss"].item(), step)
    writer.add_scalar("train/kl_loss", metrics["kl_loss"].item(), step)
    writer.add_scalar("train/reward_mean", metrics["rewards"].mean().item(), step)
    writer.add_scalar("train/reward_std", metrics["rewards"].std().item(), step)
    writer.add_scalar("train/ratio_mean", metrics["old_ratio"].mean().item(), step)
    writer.add_scalar("train/entropy", metrics["entropy"].item(), step)

In [ ]:
def maybe_log_evals(pi, writer, step, eval_metrics_steps, eval_generation_steps, gsm8k, gsm_hard, gsm_hard_samples_indices, gsm_hard_samples):
    if not step % eval_metrics_steps:
        reward_gsmk, acc_gsmk= evaluate_gsm8k(pi, gsm8k, tokenizer, REWARD_FUNCS)
        acc_gsm_hard = evaluate_gsm_hard(pi, gsm_hard, tokenizer)
        writer.add_scalar("eval/reward_gsm8k_small", reward_gsmk, step)
        writer.add_scalar("eval/acc_gsm_small", acc_gsmk, step)
        writer.add_scalar("eval/acc_gsm_hard_small", acc_gsm_hard, step)
    if not step % eval_generation_steps:
        for i, ex, completion in zip(gsm_hard_samples_indices, gsm_hard_samples, infer_batch(pi, gsm_hard_samples)):
            logger.info(f"**Q:** {ex['question']}\n\n**A:** {ex['answer']}\n\n**Gen:** {completion}")
            writer.add_text(f"samples/gsm_hard_{i}", f"**Q:** {ex['question']}\n\n**A:** {ex['answer']}\n\n**Gen:** {completion}", step)

In [ ]:
def maybe_push_model(pi, step, checkpoint_steps, model_name_tb, model_name_hf):
    if step and not step % checkpoint_steps:
        pi.save_pretrained(f"/checkpoints/{model_name_tb}/step_{step}")
        pi.push_to_hub(f"zinchse/{model_name_hf}", commit_message=f"After #{step} steps on gsm8k")

## Helper to recover currupted model
Use it to restore the model if prev train cycle was interrupted.
In case of OOM simply call `cleanup_cuda`.

In [ ]:
def recover_model(model, optimizer, remove_cache=False):
    model.enable_adapter_layers()
    model.train()
    model.config.use_cache = False
    optimizer.zero_grad(set_to_none=True)
    if remove_cache:
        if hasattr(model.base_model.model, '_cache'):
           del model.base_model.model._cache
    cleanup_cuda()

# Train-loop

In [ ]:
# from torch.utils.tensorboard import SummaryWriter
# from transformers import get_cosine_schedule_with_warmup

# model_name = config.model_name.split("/")[1]
# model_name_hf = model_name + "grpo_gsm8k"
# writer = SummaryWriter(f"{config.log_dir}/{model_name}")

# pi = model
# pi.train()
# opt = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
# scheduler = get_cosine_schedule_with_warmup(
#     opt,
#     num_warmup_steps=int(0.1 * config.num_updates),
#     num_training_steps=config.num_updates,
# )
# k = config.accumulation_steps # multiplier to convert # step to the # weights_update

# mem(f"Memory on model {model_name} with Adam")

In [ ]:
# import itertools

# try:
#     data_iter = itertools.cycle(gsm8k['train'].shuffle())
    
#     for step in range(config.num_updates * k + 1):
#         ex = next(data_iter)
#         # use it if you're at the edge of memory limits
#         gc.collect()
#         torch.cuda.empty_cache()
    
#         trajectories = generate_group(pi=pi, ex=ex, group_size=config.group_size, max_new_tokens=config.max_new_tokens)
        
#         metrics = calculate_metrics(pi=pi, trajectories=trajectories, ex=ex, eps=config.eps, beta=config.beta, accumulation_steps=config.accumulation_steps)
        
#         maybe_do_opt_step(
#             pi=pi, opt=opt, scheduler=scheduler, loss=metrics["loss"], step=step,
#             accumulation_steps=config.accumulation_steps, 
#             clip_grad_norm=config.clip_grad_norm,
#         )
        
#         maybe_log_evals(
#             pi=pi, step=step, writer=writer, gsm8k=gsm8k_small, gsm_hard=gsm_hard_small,
#             eval_metrics_steps=k*config.eval_metrics_steps,
#             eval_generation_steps=k*config.eval_generation_steps, 
#             gsm_hard_samples_indices=gsm_hard_samples_indices, 
#             gsm_hard_samples=gsm_hard_samples, 
#         )
        
#         maybe_push_model(
#             pi=pi, step=step, checkpoint_steps=k*config.checkpoint_steps, 
#             model_name_tb=model_name, model_name_hf=model_name_hf,
#         )
        
#         log_metrics(metrics=metrics, step=step, accumulation_steps=config.accumulation_steps, writer=writer)
#     logging.info("Done!")
#     pi.push_to_hub(f"zinchse/{model_name_hf}", commit_message=f"After #{step} steps on gsm8k")    

# except Exception as e:
#     logger.exception(f"Something went wrong {e}")
# finally:
#     from huggingface_hub import upload_folder, upload_file
#     upload_file(
#         path_or_fileobj="/train.log",
#         path_in_repo="train.log",
#         repo_id=f"zinchse/{model_name_hf}",
#     )
#     upload_folder(
#         folder_path="/workspace/logs",
#         path_in_repo="logs",
#         repo_id=f"zinchse/{model_name_hf}",
#     )
#     finish_vastai_pod("your_vastai_key")

# Evals
To avoid focus on the format here we will use **soft answer matching**, i.e. just matching the answer with the last written number.

**Don't forget to load checkpoint, not default Qwen!**

In [ ]:
# for tensorboard logs use
# from huggingface_hub import snapshot_download
# model_name = config.model_name.split("/")[1]
# model_name_hf = model_name + "grpo_gsm8k"
# snapshot_download(repo_id="zinchse/"+model_name_hf, local_dir="logs")

pi = model
pi.eval()

## `gsm_hard`
Does my GRPO generalize on harder questions?

In [ ]:
# initially was 0.27
with pi.disable_adapter():
    acc_gsm_hard_raw = evaluate_gsm_hard(pi, gsm_hard, tokenizer, soft_answer=False)    
    print("gsm_hard raw accuracy:", acc_gsm_hard_raw)

In [ ]:
# initially was 35, 37.7 on some 800-steps run, 36 on 1200-steps run
acc_gsm_hard_rlvr = evaluate_gsm_hard(pi, gsm_hard, tokenizer, soft_answer=False)
print("gsm_hard RLVR accuracy:", acc_gsm_hard_rlvr)

In [ ]:
# initially was 0.27
with pi.disable_adapter():
    acc_gsm_hard_raw_soft = evaluate_gsm_hard(pi, gsm_hard, tokenizer, soft_answer=True)    
    print("gsm_hard raw accuracy soft:", acc_gsm_hard_raw_soft)

## `gsm8k`

In [ ]:
# initially was 0.3
with pi.disable_adapter():
    reward_gsm8k_raw, acc_gsm8k_raw = evaluate_gsm8k(pi, gsm8k['test'], tokenizer, REWARD_FUNCS)
print("gsm8k raw reward:", reward_gsm8k_raw, " accuracy:", acc_gsm8k_raw)

In [ ]:
# initially was 0.71, 0.75 on some 800-steps run, 72 on 1200-steps run
pi.enable_adapter_layers()
reward_gsm8k_rlvr, acc_gsm8k_rlvr = evaluate_gsm8k(pi, gsm8k['test'], tokenizer, REWARD_FUNCS)
print("gsm8k RLVR reward:", reward_gsm8k_rlvr, " accuracy:", acc_gsm8k_rlvr)

In [ ]:
# initially was ~0.56
with pi.disable_adapter():
    reward_gsm8k_raw_soft, acc_gsm8k_raw_soft = evaluate_gsm8k(pi, gsm8k['test'].select(range(128*7-1, 128*8)), tokenizer, REWARD_FUNCS, soft_answer=True)
print("gsm8k raw reward:", reward_gsm8k_raw_soft, " accuracy (soft):", acc_gsm8k_raw_soft)

## saving

In [ ]:
import json

results = {
    "model": config.model_name,
    "training": {
        "dataset": "openai/gsm8k",
        "algorithm": "GRPO from scratch",
        "num_updates": config.num_updates,
        "group_size": config.group_size,
        "lora_rank": config.lora_rank,
        "lr": config.lr,
        "beta": config.beta,
    },
    "eval": {
        "gsm8k_raw": acc_gsm8k_raw,
        "gsm8k_rlvr": acc_gsm8k_rlvr,
        "gsm8k_raw_soft": acc_gsm8k_raw_soft,
        "gsm_hard_raw": acc_gsm_hard_raw,
        "gsm_hard_rlvr": acc_gsm_hard_rlvr,
        "gsm_hard_raw_soft": acc_gsm_hard_raw_soft,
    }
}

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

## plotting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(2)
width = 0.25

ax.bar(x - width, [results["eval"]["gsm8k_raw"], results["eval"]["gsm_hard_raw"]], width, label="Base model (strict match)", color="#4a4a6a")
ax.bar(x,         [results["eval"]["gsm8k_raw_soft"], results["eval"]["gsm_hard_raw_soft"]],                           width, label="Base model (soft match)", color="#7a7a9a")
ax.bar(x + width, [results["eval"]["gsm8k_rlvr"], results["eval"]["gsm_hard_rlvr"]], width, label="After GRPO (strict match)", color="#e86a3a")

ax.set_xticks(x)
ax.set_xticklabels(["GSM8K (eval)", "GSM-Hard (OOD)"], fontsize=13)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(fontsize=11)
ax.set_title("GRPO Generalization: trained on GSM8K, evaluated on GSM-Hard", fontsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results.png", dpi=150)
plt.show()